# DWD 层 — 清洗统一 + 星型模型

ODS Yellow + Green → 统一字段口径 → 清洗 → 衍生列 → 写出 `fact_trips`

同时构建维度表：`dim_zone`、`dim_date`

## 1. SparkSession（启用动态分区覆盖）

In [1]:
import os
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, TimestampType

spark = (
    SparkSession.builder
    .appName("TLC-DWD-Build")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.driver.memory", "4g")
    # 动态分区覆盖：INSERT OVERWRITE 只覆盖本次写入涉及的分区，保证幂等重跑
    .config("spark.sql.sources.partitionOverwriteMode", "dynamic")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

BASE        = os.path.expanduser("~/tlc-spark-pipeline")
ODS_Y       = f"{BASE}/data/warehouse/ods/yellow"
ODS_G       = f"{BASE}/data/warehouse/ods/green"
ODS_FHV     = f"{BASE}/data/warehouse/ods/fhv_hv"
DWD_FACT    = f"{BASE}/data/warehouse/dwd/fact_trips"
DWD_ZONE    = f"{BASE}/data/warehouse/dwd/dim_zone"
DWD_DATE    = f"{BASE}/data/warehouse/dwd/dim_date"

TARGET_MONTH = os.environ.get("TARGET_MONTH", "2024-03")
yr, mo = int(TARGET_MONTH.split("-")[0]), int(TARGET_MONTH.split("-")[1])
print(f"TARGET_MONTH={TARGET_MONTH}")

print(f"Spark {spark.version} ready | TARGET_MONTH={TARGET_MONTH}")

## 2. 统一字段口径

黄色多 `Airport_fee`，绿色多 `trip_type` / `ehail_fee`。统一为相同列集合，差异字段补 null。

In [2]:
yellow = spark.read.parquet(ODS_Y).filter(
    (F.year("pickup_datetime") == yr) & (F.month("pickup_datetime") == mo))
green  = spark.read.parquet(ODS_G).filter(
    (F.year("pickup_datetime") == yr) & (F.month("pickup_datetime") == mo))
fhv    = spark.read.parquet(ODS_FHV).filter(
    (F.year("pickup_datetime") == yr) & (F.month("pickup_datetime") == mo))

# 黄色：补绿色独有列
yellow_unified = (
    yellow
    .withColumn("trip_type",  F.lit(None).cast("int"))      # 绿色才有
    .withColumn("Airport_fee", F.col("Airport_fee"))         # 黄色已有
)

# 绿色：补黄色独有列，ehail_fee 全 null 不保留
green_unified = (
    green
    .withColumn("Airport_fee", F.lit(0.0).cast("double"))   # 绿色不去机场，补 0
    .drop("ehail_fee")                                       # 永远 null，丢弃
)

# FHV HV：字段映射到统一口径
fhv_unified = (
    fhv
    .withColumnRenamed("base_passenger_fare", "fare_amount")
    .withColumnRenamed("trip_miles",          "trip_distance")
    .withColumnRenamed("tips",                 "tip_amount")
    .withColumnRenamed("tolls",                "tolls_amount")
    .withColumnRenamed("airport_fee",          "Airport_fee")
    .withColumn("passenger_count",        F.lit(None).cast("int"))
    .withColumn("payment_type",            F.lit(None).cast("int"))
    .withColumn("improvement_surcharge",   F.lit(0.0).cast("double"))
    .withColumn("mta_tax",                 F.lit(0.0).cast("double"))
    .withColumn("trip_type",               F.lit(None).cast("int"))
    # 计算 total_amount：基础票价 + 过路费 + 拥堵费 + 机场费 + 小费
    .withColumn("total_amount",
        F.coalesce(F.col("fare_amount"),        F.lit(0)) +
        F.coalesce(F.col("tolls_amount"),       F.lit(0)) +
        F.coalesce(F.col("congestion_surcharge"), F.lit(0)) +
        F.coalesce(F.col("Airport_fee"),        F.lit(0)) +
        F.coalesce(F.col("tip_amount"),         F.lit(0)))
)

# 取三者共同列（按黄色顺序）
common_cols = [c for c in yellow_unified.columns
               if c in green_unified.columns and c in fhv_unified.columns]

combined = (
    yellow_unified.select(common_cols)
    .union(green_unified.select(common_cols))
    .union(fhv_unified.select(common_cols))
)

print(f"Combined rows (pre-clean): {combined.count():,}")
print(f"Columns: {len(common_cols)} → {common_cols}")


## 3. 清洗 + 衍生列

In [3]:
from pyspark.sql.functions import col, unix_timestamp, round as spark_round, hour, month, year, when, dayofweek

cleaned = (
    combined
    .filter(col("fare_amount")    > 0)
    .filter(col("trip_distance")  > 0)
    .filter(col("dropoff_datetime") > col("pickup_datetime"))
    .filter(col("PULocationID").isNotNull())
    .filter(col("DOLocationID").isNotNull())
    .filter((col("passenger_count").between(1, 6)) | col("passenger_count").isNull())
    .filter(col("pickup_datetime") >= "2024-01-01")
    .filter(col("pickup_datetime") <  "2024-04-01")
)

fact = (
    cleaned
    .withColumn("trip_duration_min",
        spark_round((unix_timestamp("dropoff_datetime") - unix_timestamp("pickup_datetime")) / 60, 2))
    .withColumn("avg_speed_mph",
        spark_round(col("trip_distance") / (col("trip_duration_min") / 60), 2))
    .withColumn("fare_per_mile",
        spark_round(col("fare_amount") / col("trip_distance"), 2))
    # 金额口径统一：surcharge_total 汇总各类附加费
    .withColumn("surcharge_total",
        spark_round(
            F.coalesce(col("improvement_surcharge"), F.lit(0)) +
            F.coalesce(col("mta_tax"), F.lit(0)) +
            F.coalesce(col("congestion_surcharge"), F.lit(0)) +
            F.coalesce(col("Airport_fee"), F.lit(0)), 2))
    .withColumn("time_of_day",
        when(hour("pickup_datetime").between(6,  9),  "morning_rush")
        .when(hour("pickup_datetime").between(10, 15), "midday")
        .when(hour("pickup_datetime").between(16, 19), "evening_rush")
        .when(hour("pickup_datetime").between(20, 23), "night")
        .otherwise("late_night"))
    .withColumn("payment_label",
        when(col("payment_type") == 1, "credit_card")
        .when(col("payment_type") == 2, "cash")
        .when(col("payment_type") == 3, "no_charge")
        .otherwise("other"))
    .withColumn("pickup_year",  year("pickup_datetime"))
    .withColumn("pickup_month", month("pickup_datetime"))
    .withColumn("pickup_hour",  hour("pickup_datetime"))
    .withColumn("day_of_week",  dayofweek("pickup_datetime"))
    .filter(col("avg_speed_mph").between(0, 100))
    .filter(col("trip_duration_min").between(1, 180))
)

fact_count = fact.count()
print(f"fact_trips rows: {fact_count:,}")
fact.groupBy("source_type", "pickup_month").count().orderBy("source_type", "pickup_month").show()

## 4. 写出 fact_trips（动态分区覆盖）

In [4]:
%%time
# partitionOverwriteMode=dynamic：只覆盖本次涉及的分区，重跑某个月不影响其他月
(
    fact
    .write
    .partitionBy("source_type", "pickup_year", "pickup_month")
    .mode("overwrite")
    .parquet(DWD_FACT)
)
print(f"fact_trips written to: {DWD_FACT}")

## 5. 维度表：dim_zone

In [5]:
# 从 fact 中提取所有出现的 LocationID，构建区域维度表
pu_zones = fact.select(F.col("PULocationID").alias("location_id"))
do_zones = fact.select(F.col("DOLocationID").alias("location_id"))
dim_zone = pu_zones.union(do_zones).distinct().orderBy("location_id")

dim_zone.write.mode("overwrite").parquet(DWD_ZONE)
print(f"dim_zone: {dim_zone.count()} zones → {DWD_ZONE}")

## 6. 维度表：dim_date

In [6]:
from pyspark.sql.functions import to_date, dayofweek, weekofyear, quarter

dim_date = (
    fact
    .select(to_date("pickup_datetime").alias("date"))
    .distinct()
    .withColumn("year",        year("date"))
    .withColumn("month",       month("date"))
    .withColumn("day",         F.dayofmonth("date"))
    .withColumn("day_of_week", dayofweek("date"))
    .withColumn("week",        weekofyear("date"))
    .withColumn("quarter",     quarter("date"))
    .withColumn("is_weekend",  (dayofweek("date").isin(1, 7)).cast("boolean"))
    .orderBy("date")
)

dim_date.write.mode("overwrite").parquet(DWD_DATE)
print(f"dim_date: {dim_date.count()} dates → {DWD_DATE}")
dim_date.show(5)

spark.stop()
print("DWD build complete.")